# Azure Code Agent - Multi-Agent System

## Setup

Make sure you have:
1. Installed dependencies: `pip install -r requirements.txt`
2. For **Local LLM**: LM Studio running with your model loaded
3. For **Azure OpenAI**: `.env` file with Azure credentials

In [1]:
# Add parent directory to path so we can import azure_agent
import sys
from pathlib import Path

# Add the parent directory to Python path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Import the new multi-agent system
from azure_agent import run, run_interactive

print("✅ Multi-Agent System loaded!")

✅ Multi-Agent System loaded!


## Quick Start - Run with prompt.md

The simplest way to use the system. It will:
1. Read your project spec from `prompt.md`
2. Run all 5 agents in sequence
3. Generate complete, working code in `./output/`
4. Validate and fix any issues automatically

In [2]:
# Run the complete multi-agent workflow with Local LLM
results = run(
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    local_api_base="http://localhost:1234/v1",
)

# Results summary
print(f"\n✅ Final Status: {results['final_status']}")
print(f"📊 Quality Score: {results['validation_result'].get('score', 0)}/100")
print(f"📁 Output: {results['output_directory']}")

🏠 Using Local LLM
   API Base: http://localhost:1234/v1
   Model: qwen/qwen3-coder-30b
📝 Logging to: /Users/Khaled.Alabsi/projects/coding-agent/output/logs/agent_log_20251108_182937.json
📂 Step results: /Users/Khaled.Alabsi/projects/coding-agent/output/logs/workflow_steps_20251108_182937
🚀 MULTI-AGENT CODING WORKFLOW

📁 Output Directory: /Users/Khaled.Alabsi/projects/coding-agent/output
   💾 Saved to: 0_user_prompt.md

IMPLEMENTATION (Coder-driven with tools)

⏳ Coder: Waiting for LLM response...
✅ Coder: Received LLM response (522 chars)

Iteration 1/200

🤖 Coder: TOOL: ENHANCE_PROMPT
INPUT:
# Project: very nice looking personal website
Create a personal website that is visually appealing, user-friendly, and showcases my portfolio, blog, and contact information. The design should be modern and responsive, ensuring it looks great on both desktop and mobile devices. Include sections for an about me, projects, blog posts, and a contact form. Use a color scheme that reflects my personalit

## Custom Prompt - Direct Input

Instead of using `prompt.md`, you can provide your prompt directly:

In [ ]:
# Run with a custom prompt
results = run(
    prompt="Create a simple calculator app with React and TypeScript. Include basic operations: add, subtract, multiply, divide.",
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    output_dir="./output/calculator-app"
)

print(f"\nStatus: {results['final_status']}")
print(f"Location: {results['output_directory']}")

## Load Prompt from File

Use a custom prompt file instead of `prompt.md`:

In [ ]:
# Run with a custom prompt file
results = run(
    prompt_file="my-project-spec.md",
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    output_dir="./output/my-project"
)

## Advanced Options

Fine-tune the workflow with advanced options:

In [ ]:
# Advanced configuration
results = run(
    prompt="Create a modern portfolio website with dark mode toggle",
    
    # LLM Configuration
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    
    # Output
    output_dir="./output/portfolio",
    
    # Workflow Options
    skip_prompt_enhancement=False,  # Set True to skip prompt enhancement
    skip_plan_enhancement=False,    # Set True to skip plan enhancement
    max_fix_iterations=5,           # More retries for higher quality
    
    # Generation Parameters
    temperature=0.7,     # Higher = more creative, lower = more focused
    max_tokens=2000,     # Max tokens per agent response
    context_window=8000  # Context window size
)

# Detailed results
print("\n" + "="*70)
print("WORKFLOW RESULTS")
print("="*70)
print(f"Enhanced Prompt: {results['enhanced_prompt'][:200]}...")
print(f"\nFix Iterations Used: {results['fix_iterations']}/{results.get('max_fix_iterations', 3)}")
print(f"Final Status: {results['final_status']}")
print(f"Quality Score: {results['validation_result'].get('score', 0)}/100")
print(f"\nOutput Directory: {results['output_directory']}")

## Using Azure OpenAI

Switch to Azure OpenAI (requires Azure credentials):

In [ ]:
# Run with Azure OpenAI (uncomment and configure)
# results = run(
#     prompt="Create a REST API with Node.js and Express",
#     use_local_llm=False,
#     azure_api_key="your-api-key",        # Or set AZURE_OPENAI_API_KEY env var
#     azure_endpoint="https://...",        # Or set AZURE_OPENAI_ENDPOINT
#     azure_deployment="gpt-4",            # Or set AZURE_OPENAI_DEPLOYMENT
#     output_dir="./output/api"
# )

## Inspect Results

Examine the workflow results in detail:

In [ ]:
# View enhanced prompt
print("ENHANCED PROMPT:")
print("="*70)
print(results['enhanced_prompt'])
print()

In [ ]:
# View the execution plan
print("EXECUTION PLAN:")
print("="*70)
print(results['enhanced_plan'])
print()

In [ ]:
# View validation results
validation = results['validation_result']

print("VALIDATION RESULTS:")
print("="*70)
print(f"Status: {validation.get('status', 'UNKNOWN')}")
print(f"Score: {validation.get('score', 0)}/100\n")

passed = validation.get('passed_checks', [])
if passed:
    print(f"✅ Passed Checks ({len(passed)}):")
    for check in passed:
        print(f"   • {check}")

failed = validation.get('failed_checks', [])
if failed:
    print(f"\n❌ Failed Checks ({len(failed)}):")
    for check in failed:
        print(f"   • {check}")

suggestions = validation.get('suggestions', [])
if suggestions:
    print(f"\n💡 Suggestions ({len(suggestions)}):")
    for suggestion in suggestions:
        print(f"   • {suggestion}")

In [ ]:
# List generated files
import os
from pathlib import Path

output_path = Path(results['output_directory'])
if output_path.exists():
    print("GENERATED FILES:")
    print("="*70)
    for file in sorted(output_path.rglob("*")):
        if file.is_file() and not file.name.startswith('.'):
            rel_path = file.relative_to(output_path)
            size = file.stat().st_size
            print(f"  {rel_path} ({size:,} bytes)")
else:
    print(f"Output directory not found: {output_path}")

## Interactive Mode

Run in interactive mode to enter your prompt directly:

In [ ]:
# This will prompt you to enter your project description
# run_interactive()

## Manual Workflow Control

For complete control, you can run agents individually:

In [ ]:
# Manual workflow control
from azure_agent.config import AgentConfig
from azure_agent.core import AgentOrchestrator

# Create config
config = AgentConfig(
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    local_api_base="http://localhost:1234/v1"
)

# Create orchestrator
orchestrator = AgentOrchestrator(config)

print("✅ Orchestrator created")
print(f"Output directory: {config.output_dir}")

In [ ]:
# Step 1: Enhance prompt
user_prompt = "Create a weather app"
enhanced = orchestrator.prompt_enhancer.enhance_prompt(user_prompt)
print(f"\nEnhanced: {enhanced}")

In [ ]:
# Step 2: Create plan
plan = orchestrator.planner.create_plan(enhanced)
print(f"\nPlan created ({len(plan)} chars)")

In [ ]:
# Step 3: Enhance plan
enhanced_plan = orchestrator.plan_enhancer.enhance_plan(plan)
print(f"\nEnhanced plan created ({len(enhanced_plan)} chars)")

In [ ]:
# Step 4: Execute code
implementation = orchestrator.coder.execute_plan(enhanced_plan)
print(f"\nImplementation complete")

In [ ]:
# Step 5: Validate
validation = orchestrator.validator.validate_results(enhanced_plan, implementation)
print(f"\nValidation: {validation.get('status')}")
print(f"Score: {validation.get('score')}/100")

## Examples

Common use cases:

In [ ]:
# Example 1: Todo App
results = run(
    prompt="Create a todo app with React and TypeScript",
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    output_dir="./output/todo-app"
)

In [ ]:
# Example 2: REST API
results = run(
    prompt="Create a REST API for a blog with Node.js, Express, and PostgreSQL",
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    output_dir="./output/blog-api"
)

In [ ]:
# Example 3: Data Processing Script
results = run(
    prompt="Create a Python script that processes CSV files and generates visualizations",
    use_local_llm=True,
    local_model="qwen/qwen3-coder-30b",
    output_dir="./output/data-processor"
)

## Tips & Tricks

### Performance
- Use `skip_prompt_enhancement=True` for simple, well-defined tasks
- Use `skip_plan_enhancement=True` to speed up the workflow
- Lower `temperature` (0.3-0.5) for more deterministic output
- Higher `temperature` (0.7-0.9) for more creative solutions

### Quality
- Increase `max_fix_iterations` (5-10) for higher quality
- Use more detailed prompts for better results
- Review the enhanced prompt to ensure it matches your intent

### Debugging
- Check `workflow_results.json` in the output directory
- Review validation results for specific issues
- Use manual workflow control to debug individual agents

## Troubleshooting

### Local LLM Issues
- **Connection refused**: Start LM Studio server
- **Slow responses**: Use smaller model or reduce context_window
- **Model not found**: Check exact model name in LM Studio

### Quality Issues
- **Low validation score**: Increase `max_fix_iterations`
- **Missing files**: Check the plan and coder output
- **Broken imports**: The validator should catch and fix these

### General
- **Module not found**: Check pip install ran successfully
- **Permission errors**: Check output directory permissions
- **Memory errors**: Reduce context_window or use smaller model

---

## Context & Token Analyzer

Diagnose truncated responses and optimize context settings

In [ ]:
from utils.llm_tester import test_llm_context

print("✅ LLM Tester loaded!")

In [ ]:
# Stress test - push both to limits
test_llm_context(max_tokens=32000, context_window=262144)

In [ ]:
# REAL context window stress test - with YOUR test values
test_context_window = 262144
test_max_tokens = 65500

# Calculate max prompt size (leave room for response)
max_prompt_tokens = test_context_window - test_max_tokens  # 196,644 tokens
target_chars = max_prompt_tokens * 4  # ~786k chars

# Generate prompt close to limit
base_text = "Analyze this code in detail. "
repeats = target_chars // len(base_text)
huge_prompt = base_text * repeats

print(f"Testing with: context_window={test_context_window:,}, max_tokens={test_max_tokens:,}")
print(f"Max prompt size: {max_prompt_tokens:,} tokens (~{target_chars:,} chars)")
print(f"Generated prompt: ~{len(huge_prompt):,} chars (~{len(huge_prompt)//4:,} tokens)")
print("\nSending to model...")
print()

result = test_llm_context(
    prompt=huge_prompt,
    max_tokens=test_max_tokens,
    context_window=test_context_window
)

if 'error' in result:
    print(f"\n❌ CONTEXT WINDOW TEST FAILED")
    print(f"Model does NOT support {test_context_window:,} context window")
else:
    print(f"\n✅ CONTEXT WINDOW TEST PASSED")
    print(f"Model accepts {result['prompt_tokens']:,} token prompt")

In [ ]:
# Override only if you need to test different values
# test_llm_context(max_tokens=10000)  # Uncomment to test with different max_tokens